In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

# =========================
# 1. 데이터 로드
# =========================
train = pd.read_csv(r"C:\Users\a0916\dartb\DartB_Titanic\train.csv")
test = pd.read_csv(r"C:\Users\a0916\dartb\DartB_Titanic\test.csv")

# 제출 파일 만들 때 PassengerId가 필요하므로 따로 저장
test_passenger_id = test['PassengerId'].copy()

# -------------------------
# 보고서에 쓰기 좋은 기본 확인용 출력
# (가이드라인 1번, 4번 실습 반영)
# - 결측치가 어디에 있는지 먼저 확인
# - 왜도 처리가 필요한 대표 컬럼(Fare)의 왜도를 숫자로 확인
# -------------------------
print("전처리 전 결측치 개수")
print(train[['Age', 'Cabin', 'Embarked']].isnull().sum())
print(f"\n전처리 전 Fare 왜도: {train['Fare'].skew():.4f}")


# =========================
# 2. 전처리에 필요한 함수
# =========================
def extract_title(name_series):
    title = name_series.str.extract(r' ([A-Za-z]+)\.', expand=False)
    title = title.replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr',
                           'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    title = title.replace(['Mlle', 'Ms'], 'Miss')
    title = title.replace('Mme', 'Mrs')
    return title


# =========================
# 3. 기본 피처 생성
# =========================
for df in [train, test]:
    # [가이드라인 6번 반영]
    # 이름에서 호칭 추출: 사회적 지위, 연령대 정보가 어느 정도 담겨 있다고 보고 사용
    df['Title'] = extract_title(df['Name'])

    # 가족 규모
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

    # 혼자인지 여부
    # FamilySize를 그대로 쓰는 것 외에, 혼자 탔는지를 따로 보는 것도 의미가 있다고 판단
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    # Cabin 자체는 결측이 많아서 그대로 쓰기 애매하지만,
    # 객실 정보가 있었는지 여부는 정보가 될 수 있어서 0/1로만 남김
    df['CabinKnown'] = df['Cabin'].notna().astype(int)


# =========================
# 4. 결측치 처리
# =========================

# [가이드라인 1번 반영]
# Embarked는 범주형 변수라 최빈값으로 채움
# train 기준으로 값을 정해두고 test에도 같은 기준을 적용
embarked_mode = train['Embarked'].mode()[0]
train['Embarked'] = train['Embarked'].fillna(embarked_mode)
test['Embarked'] = test['Embarked'].fillna(embarked_mode)

# [가이드라인 1번 반영]
# Fare는 test에 결측치가 있을 수 있으므로,
# Pclass별 중앙값 -> 전체 중앙값 순서로 채움
# 기존 코드처럼 NaN을 바로 0처럼 처리하면
# '결측치'와 '실제 0'이 섞여버려서 해석이 애매해질 수 있음
fare_by_pclass = train.groupby('Pclass')['Fare'].median()
fare_median = train['Fare'].median()

for df in [train, test]:
    df['Fare'] = df['Fare'].fillna(df['Pclass'].map(fare_by_pclass))
    df['Fare'] = df['Fare'].fillna(fare_median)

# [가이드라인 1번 반영]
# Age는 한 번에 전체 평균으로 채우기보다
# 1) Title별 중앙값
# 2) 그래도 비면 Sex + Pclass 기준 중앙값
# 3) 마지막으로 전체 중앙값
# 순서로 채움
# 이렇게 하면 Master, Miss 같은 그룹의 특성을 조금 더 살릴 수 있음
age_by_title = train.groupby('Title')['Age'].median()
age_by_sex_pclass = train.groupby(['Sex', 'Pclass'])['Age'].median()
age_median = train['Age'].median()

def fill_age(df):
    df = df.copy()

    # 1차: Title 기준
    df['Age'] = df['Age'].fillna(df['Title'].map(age_by_title))

    # 2차: Sex + Pclass 기준
    sex_pclass_key = df[['Sex', 'Pclass']].apply(tuple, axis=1)
    df['Age'] = df['Age'].fillna(sex_pclass_key.map(age_by_sex_pclass))

    # 3차: 전체 중앙값
    df['Age'] = df['Age'].fillna(age_median)

    return df

train = fill_age(train)
test = fill_age(test)


# =========================
# 5. 왜도 / 이상치 처리
# =========================

# [가이드라인 5번 반영]
# Fare는 상위 일부 값이 매우 커서 분포가 한쪽으로 길게 늘어져 있음.
# 여기서는 아예 행을 지우기보다, train 기준 상위 1% 값을 상한으로 두고 잘라냄(clip).
# 데이터가 많지 않아서 제거보다는 완화하는 쪽이 더 무난하다고 봤음.
fare_upper = train['Fare'].quantile(0.99)

train['Fare'] = train['Fare'].clip(upper=fare_upper)
test['Fare'] = test['Fare'].clip(upper=fare_upper)

# [가이드라인 4번 반영]
# Fare는 오른쪽 꼬리가 긴 분포라 log1p 변환 적용
# 기존 코드의 np.log(x) 대신 np.log1p(x)를 쓰면 0도 자연스럽게 처리 가능
train['Fare'] = np.log1p(train['Fare'])
test['Fare'] = np.log1p(test['Fare'])

print(f"\n전처리 후 Fare 왜도: {train['Fare'].skew():.4f}")


# =========================
# 6. 학습용 데이터 정리
# =========================

# [가이드라인 6번 반영]
# 불필요한 고유값 컬럼 제거
# PassengerId: 식별자
# Name, Ticket: 그대로 넣으면 차원이 불필요하게 늘어나기 쉬움
# Cabin: 결측이 너무 많아서 CabinKnown만 남기고 제거
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
train_model = train.drop(columns=drop_cols)
test_model = test.drop(columns=drop_cols)

# [가이드라인 2번 반영]
# Sex, Embarked, Title, Pclass는 범주형으로 보고 원-핫 인코딩
# Pclass는 숫자처럼 보여도 실제로는 등급 개념의 범주형에 가까워서 문자열로 바꿔 인코딩
train_model['Pclass'] = train_model['Pclass'].astype(str)
test_model['Pclass'] = test_model['Pclass'].astype(str)

# train/test를 따로 get_dummies 하면 컬럼이 어긋날 수 있어서
# 먼저 합친 뒤 한 번에 인코딩하고 다시 나눔
combined = pd.concat(
    [train_model.drop(columns='Survived'), test_model],
    axis=0
)

combined = pd.get_dummies(
    combined,
    columns=['Sex', 'Embarked', 'Title', 'Pclass'],
    drop_first=False
)

X = combined.iloc[:len(train_model)].copy()
X_test = combined.iloc[len(train_model):].copy()
y = train_model['Survived']

print(f"\n전처리 후 남은 결측치 수: {X.isnull().sum().sum()}")

# [가이드라인 3번 반영]
# 스케일링은 적용하지 않음.
# 이유: 랜덤포레스트는 거리 기반 모델이 아니라서
# 선형모델이나 KNN처럼 스케일에 크게 민감하지 않기 때문.
# 그래서 이번에는 스케일링보다 결측치 처리와 피처 생성 쪽이 더 중요하다고 판단함.


# =========================
# 7. 모델 선택 / 검증
# =========================

# [가이드라인 7번 반영]
# train/valid를 한 번만 나누는 것보다
# Stratified K-Fold로 여러 번 검증하는 쪽이 결과가 덜 흔들림.
# 타이타닉은 생존/사망 비율 차이가 있으므로 Stratified 방식 사용
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# [가이드라인 8번, 10번 반영]
# 타이타닉은 변수 간 관계가 단순 선형이라고 보기 어려워서 트리 기반 모델 선택
# 과적합을 조금 줄이려고 max_depth와 min_samples_leaf를 제한
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=2,
    random_state=42
)

# [가이드라인 9번 반영]
# Accuracy만 보면 놓치는 부분이 있을 수 있어서 F1도 같이 확인
cv_result = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring={'accuracy': 'accuracy', 'f1': 'f1'},
    n_jobs=-1
)

print("\n교차검증 결과")
print(f"평균 Accuracy: {cv_result['test_accuracy'].mean():.4f}")
print(f"평균 F1-score: {cv_result['test_f1'].mean():.4f}")


# =========================
# 8. 전체 학습 후 예측
# =========================
model.fit(X, y)

# 중요 변수 확인
# 모델이 어떤 변수에 더 의존하는지 간단히 보기 위해 출력
feature_importance = pd.Series(model.feature_importances_, index=X.columns)
feature_importance = feature_importance.sort_values(ascending=False)

print("\n중요 변수 상위 10개")
print(feature_importance.head(10))

# 예측 및 저장
submission = pd.DataFrame({
    'PassengerId': test_passenger_id,
    'Survived': model.predict(X_test)
})

submission.to_csv('improved_submission.csv', index=False)
print("\nimproved_submission.csv 저장 완료")

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'